# Experiment 5.3.1 — Factorized gate causal attribution

Analysis-only notebook. Relative10 is treated as an ordered-trajectory preservation ceiling, Hidden WholeCount is the phase-blind internalization probe, reset/shuffle plus an ordered-trained transfer probe provide causal attribution, and native accumulator BA is the deployment outcome.


In [ ]:
from __future__ import annotations
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'scripts').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate writingRing repository root')

ROOT = find_repo_root()
OUT = ROOT / 'notebooks/artifacts/experiment_5_3_1_factorized_gate_attribution/factorized_gate_causal_attribution_v1'
manifest = json.loads((OUT / 'manifest.json').read_text(encoding='utf-8'))
runs = pd.read_csv(OUT / 'runs.csv')
histories = pd.read_csv(OUT / 'histories.csv')
probes = pd.read_csv(OUT / 'probe_runs.csv')
phase_probes = pd.read_csv(OUT / 'phase_probe_runs.csv')
ablations = pd.read_csv(OUT / 'ablation_runs.csv')
sensitivity = pd.read_csv(OUT / 'history_sensitivity_runs.csv')
local = pd.read_csv(OUT / 'local_reference.csv')
ORDER = ['factorized_ff_gate','factorized_lif22','factorized_lif242','factorized_rsnn22']
assert manifest['expected_runs'] == 20 and len(runs) == 20
assert runs[['condition','seed']].drop_duplicates().shape[0] == 20
print(manifest['protocol_version'], OUT)


## 1. Preservation and phase-blind internalization

`hidden_relative10_ordered` is only a preservation ceiling because the probe receives explicit temporal order. The primary representation metric is `hidden_whole_count - local_whole_count`.


In [ ]:
local_summary = local.groupby('probe_type').test_ba.agg(['mean','std']).reset_index()
local_wc = float(local_summary.loc[local_summary.probe_type == 'local_whole_count', 'mean'].iloc[0])
local_rel10 = float(local_summary.loc[local_summary.probe_type == 'local_relative10_ordered', 'mean'].iloc[0])
pw = probes.pivot_table(index=['condition','seed'], columns='probe_type', values='test_ba').reset_index()
pw['internalization_gain'] = pw['hidden_whole_count'] - local_wc
pw['preservation_loss'] = pw['hidden_relative10_ordered'] - local_rel10
matrix = pw.groupby('condition').agg(
    hidden_whole_count=('hidden_whole_count','mean'),
    hidden_whole_count_sd=('hidden_whole_count','std'),
    hidden_relative10=('hidden_relative10_ordered','mean'),
    hidden_relative10_sd=('hidden_relative10_ordered','std'),
    internalization_gain=('internalization_gain','mean'),
    preservation_loss=('preservation_loss','mean'),
).reindex(ORDER).reset_index()
display(local_summary)
display(matrix)

fig, ax = plt.subplots(figsize=(10.5, 5.5))
x = np.arange(len(ORDER)); width = 0.34
ax.bar(x - width/2, matrix.hidden_whole_count, width, yerr=matrix.hidden_whole_count_sd, capsize=3, label='Hidden WholeCount')
ax.bar(x + width/2, matrix.hidden_relative10, width, yerr=matrix.hidden_relative10_sd, capsize=3, label='Hidden Relative10 ceiling')
ax.axhline(local_wc, linestyle='--', linewidth=1.2, label='Local WholeCount')
ax.axhline(local_rel10, linestyle=':', linewidth=1.2, label='Local Relative10 ceiling')
ax.set_xticks(x); ax.set_xticklabels(ORDER, rotation=20, ha='right')
ax.set_ylabel('Test balanced accuracy')
ax.set_title('Preservation ceiling versus phase-blind internalization')
ax.legend(); fig.tight_layout(); plt.show()


## 2. Paired contribution decomposition

The trained-model contrasts isolate content remapping, passive short history, passive long history, and learned recurrence.


In [ ]:
def paired(frame: pd.DataFrame, metric: str, left: str, right: str) -> pd.DataFrame:
    wide = frame[frame.condition.isin([left,right])].pivot(index='seed', columns='condition', values=metric).dropna()
    return pd.DataFrame({'seed': wide.index, 'comparison': f'{left} - {right}', 'difference': wide[left] - wide[right]})

wc_frame = pw[['condition','seed','hidden_whole_count']].copy()
native_frame = runs[['condition','seed','native_test_balanced_accuracy']].copy()
pairs = [
    ('factorized_lif22','factorized_ff_gate'),
    ('factorized_lif242','factorized_ff_gate'),
    ('factorized_rsnn22','factorized_lif22'),
]
wc_effects = pd.concat([paired(wc_frame, 'hidden_whole_count', a, b) for a,b in pairs], ignore_index=True)
native_effects = pd.concat([paired(native_frame, 'native_test_balanced_accuracy', a, b) for a,b in pairs], ignore_index=True)
wc_summary = wc_effects.groupby('comparison').difference.agg(['mean','std','count']).reset_index()
native_summary = native_effects.groupby('comparison').difference.agg(['mean','std','count']).reset_index()
display(wc_summary)
display(native_summary)

fig, ax = plt.subplots(figsize=(9.5, 4.8))
ax.bar(np.arange(len(wc_summary)), wc_summary['mean'], yerr=wc_summary['std'], capsize=4)
ax.axhline(0, linewidth=1)
ax.set_xticks(np.arange(len(wc_summary))); ax.set_xticklabels(wc_summary.comparison, rotation=20, ha='right')
ax.set_ylabel('Paired Hidden WholeCount BA difference')
ax.set_title('Incremental history contribution beyond the trained FF gate')
fig.tight_layout(); plt.show()


## 3. Causal interventions: refit accessibility versus ordered-trained transfer

`refit` asks whether an intervention remains linearly decodable after training a new Linear probe. `transfer` keeps the ordered scaler/classifier fixed and asks whether the representation retains compatible feature semantics. Temporal-shuffle replicates are averaged within each `(condition, seed)` before cross-seed summaries.


In [ ]:
ordered_reset = ablations[ablations.ablation != 'temporal_shuffle'].copy()
shuffle = ablations[ablations.ablation == 'temporal_shuffle'].groupby(['condition','seed']).agg(
    native_test_ba=('native_test_ba','mean'),
    hidden_whole_count_refit_test_ba=('hidden_whole_count_refit_test_ba','mean'),
    hidden_whole_count_transfer_test_ba=('hidden_whole_count_transfer_test_ba','mean'),
).reset_index()
shuffle['ablation'] = 'temporal_shuffle'
ab = pd.concat([
    ordered_reset[['condition','seed','ablation','native_test_ba','hidden_whole_count_refit_test_ba','hidden_whole_count_transfer_test_ba']],
    shuffle,
], ignore_index=True)
ab_summary = ab.groupby(['condition','ablation']).agg(
    native_mean=('native_test_ba','mean'), native_sd=('native_test_ba','std'),
    refit_mean=('hidden_whole_count_refit_test_ba','mean'), refit_sd=('hidden_whole_count_refit_test_ba','std'),
    transfer_mean=('hidden_whole_count_transfer_test_ba','mean'), transfer_sd=('hidden_whole_count_transfer_test_ba','std'),
).reset_index()
display(ab_summary)

kinds = ['ordered','state_reset','temporal_shuffle']
for mean_col, sd_col, ylabel, title in [
    ('native_mean','native_sd','Native test BA','Causal dependence of deployed accumulator'),
    ('refit_mean','refit_sd','Refit Hidden WholeCount test BA','Accessibility after intervention and refit'),
    ('transfer_mean','transfer_sd','Ordered-trained transfer test BA','Feature-semantic compatibility under intervention'),
]:
    fig, ax = plt.subplots(figsize=(10.5, 5.2)); x = np.arange(len(ORDER)); width = 0.25
    for j, kind in enumerate(kinds):
        sub = ab_summary[ab_summary.ablation == kind].set_index('condition').reindex(ORDER)
        ax.bar(x + (j-1)*width, sub[mean_col], width, yerr=sub[sd_col], capsize=3, label=kind)
    ax.set_xticks(x); ax.set_xticklabels(ORDER, rotation=20, ha='right')
    ax.set_ylabel(ylabel); ax.set_title(title); ax.legend(); fig.tight_layout(); plt.show()


## 4. Direct gate/evidence history sensitivity

These measurements compare the same current local features under normal history and zero-history intervention. The FF gate should be exactly history-insensitive.


In [ ]:
test_sensitivity = sensitivity[sensitivity.split == 'test'].copy()
sens_summary = test_sensitivity.groupby('condition').agg(
    gate_history_mae=('gate_history_mae','mean'),
    active_feature_gate_history_mae=('active_feature_gate_history_mae','mean'),
    evidence_history_mae=('evidence_history_mae','mean'),
    accumulator_history_l1_mean=('accumulator_history_l1_mean','mean'),
).reindex(ORDER).reset_index()
display(sens_summary)

for metric in ['gate_history_mae','active_feature_gate_history_mae','evidence_history_mae','accumulator_history_l1_mean']:
    fig, ax = plt.subplots(figsize=(9.5,4.5))
    ax.bar(np.arange(len(ORDER)), sens_summary[metric])
    ax.set_xticks(np.arange(len(ORDER))); ax.set_xticklabels(ORDER, rotation=20, ha='right')
    ax.set_ylabel(metric); ax.set_title(f'History sensitivity: {metric}')
    fig.tight_layout(); plt.show()


## 5. Secondary phase probes and training curves

Phase probes are diagnostics only. They do not determine whether WHEN was internalized.


In [ ]:
phase_summary = phase_probes.groupby(['condition','probe_type']).test_ba.mean().reset_index()
phase_matrix = phase_summary.pivot(index='condition', columns='probe_type', values='test_ba').reindex(ORDER)
display(phase_matrix)
fig, ax = plt.subplots(figsize=(8.5,4.2))
image = ax.imshow(phase_matrix.to_numpy(), aspect='auto')
ax.set_xticks(np.arange(len(phase_matrix.columns))); ax.set_xticklabels(phase_matrix.columns, rotation=25, ha='right')
ax.set_yticks(np.arange(len(ORDER))); ax.set_yticklabels(ORDER)
ax.set_title('Secondary relative-phase linear accessibility')
fig.colorbar(image, ax=ax, label='Test BA'); fig.tight_layout(); plt.show()

curve = histories.groupby(['condition','epoch']).val_balanced_accuracy.agg(['mean','std']).reset_index()
fig, ax = plt.subplots(figsize=(10.5,5.5))
for condition in ORDER:
    sub = curve[curve.condition == condition]
    ax.plot(sub.epoch, sub['mean'], label=condition)
    ax.fill_between(sub.epoch, sub['mean']-sub['std'], sub['mean']+sub['std'], alpha=0.15)
ax.set_xlabel('Epoch'); ax.set_ylabel('Validation balanced accuracy')
ax.set_title('Validation learning curves'); ax.legend(); fig.tight_layout(); plt.show()


## Interpretation gate

Claim history internalization only if a stateful condition beats the independently trained `factorized_ff_gate` on Hidden WholeCount, loses that advantage under state reset, preserves the Relative10 ceiling, and also improves native accumulator BA. A high Relative10 score alone is not evidence of WHEN internalization.
